# 05A — Entity Extraction (GLiNER)

Pipeline stage:

- load the clean AI block corpus from 02D (`clean_ai_blocks.parquet`, 731,989 blocks)
- run GLiNER `urchade/gliner_large-v2.1` over every block, truncated to 4,000 characters
- labels: `company`, `technology`, `government institution`, `person`, `organization`, `product`, `other`
- write mentions in resumable shards of 25,000 blocks, tracked by a shard manifest
- reuse existing shards or the merged mention table when present
- merge shards into a single raw mention table (5,113,052 mentions)

Canonicalization, filtering, and entity review happen in `05B_entities.ipynb` (`pipeline/s30_entities.py`), which keeps mentions with score ≥ 0.50 inside the de-duplicated analysis corpus.

Output directory

`output/04_entity_extraction/04A_raw_mentions/`

- `entity_mentions_raw.parquet`
- `entity_run_config.json`
- `mention_shard_manifest.csv`
- `mention_shards/part_0000_mentions.parquet`
- `mention_shards/part_0001_mentions.parquet`
- ...

In [18]:
!pip -q install pyarrow aiohttp tqdm rapidfuzz gliner

## Environment Setup

In [2]:
import os
import re
import gc
import json
import math
import time
import asyncio
import hashlib
import warnings
from typing import Dict, List, Tuple, Any, Optional
from collections import Counter

import aiohttp
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from tqdm.auto import tqdm
from rapidfuzz import fuzz
from gliner import GLiNER

from google.colab import drive

drive.mount("/content/drive")

try:
    from google.colab import userdata
except Exception:
    userdata = None

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore", category=FutureWarning)

Mounted at /content/drive


## Paths

In [ ]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"
IN_DIR_02D = f"{BASE_DIR}/output/02D_sentence_train"
OUT_DIR_04 = f"{BASE_DIR}/output/04_entity_extraction"

RAW_DIR_04A = f"{OUT_DIR_04}/04A_raw_mentions"

CLEAN_AI_BLOCKS_PARQUET = f"{IN_DIR_02D}/clean_ai_blocks.parquet"
RAW_MENTION_SHARDS_DIR = f"{RAW_DIR_04A}/mention_shards"
RAW_MENTIONS_PARQUET = f"{RAW_DIR_04A}/entity_mentions_raw.parquet"
RAW_RUN_CONFIG_JSON = f"{RAW_DIR_04A}/entity_run_config.json"
RAW_MANIFEST_CSV = f"{RAW_DIR_04A}/mention_shard_manifest.csv"

os.makedirs(RAW_DIR_04A, exist_ok=True)
os.makedirs(RAW_MENTION_SHARDS_DIR, exist_ok=True)

assert os.path.exists(CLEAN_AI_BLOCKS_PARQUET), f"Missing clean AI blocks input: {CLEAN_AI_BLOCKS_PARQUET}"

print("CLEAN_AI_BLOCKS_PARQUET:", CLEAN_AI_BLOCKS_PARQUET)
print("RAW_MENTIONS_PARQUET:", RAW_MENTIONS_PARQUET)

## Runtime and Configuration

In [ ]:
SEED = 42
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NA"

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", DEVICE)
print("gpu:", GPU_NAME)
print("torch:", torch.__version__)

FORCE_RERUN_RAW_NER = False
FORCE_REBUILD_RAW_FROM_SHARDS = False
REBUILD_RAW_FROM_SHARDS_IF_NEEDED = True

GLINER_MODEL_NAME = "urchade/gliner_large-v2.1"
GLINER_THRESHOLD = 0.45
NER_BATCH_SIZE = 192
NER_BLOCKS_PER_SHARD = 25000
MAX_BLOCK_CHARS_FOR_NER = 4000
NER_LABELS = [
    "company",
    "technology",
    "government institution",
    "person",
    "organization",
    "product",
    "other",
]

# surface normalization used for entity_surface_norm
GENERIC_TECH_TERMS = {
    "ai", "artificial intelligence", "machine learning", "deep learning",
    "generative ai", "large language model", "large language models",
    "llm", "llms", "chatbot", "chatbots", "ai model", "ai models",
    "ai tool", "ai tools", "ai system", "ai systems", "ai technology",
    "ai technologies"
}

SAFE_CANONICAL_MAP = {
    "open ai": "OpenAI",
    "chat gpt": "ChatGPT",
    "gpt 4": "GPT-4",
    "gpt4": "GPT-4",
    "gpt 4o": "GPT-4o",
    "gpt4o": "GPT-4o",
    "nvidia": "Nvidia",
    "u s": "United States",
    "u s a": "United States",
    "usa": "United States",
    "u k": "United Kingdom",
    "uk": "United Kingdom",
    "eu": "European Union",
    "a i": "AI",
    "ai": "AI",
    "a i models": "AI models",
    "a i systems": "AI systems",
    "a i tools": "AI tools",
    "a i technology": "AI technology",
    "a i technologies": "AI technologies",
}

run_config = {
    "seed": SEED,
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "force_rerun_raw_ner": FORCE_RERUN_RAW_NER,
    "force_rebuild_raw_from_shards": FORCE_REBUILD_RAW_FROM_SHARDS,
    "rebuild_raw_from_shards_if_needed": REBUILD_RAW_FROM_SHARDS_IF_NEEDED,
    "gliner_model_name": GLINER_MODEL_NAME,
    "gliner_threshold": GLINER_THRESHOLD,
    "ner_labels": NER_LABELS,
    "ner_batch_size": NER_BATCH_SIZE,
    "ner_blocks_per_shard": NER_BLOCKS_PER_SHARD,
    "max_block_chars_for_ner": MAX_BLOCK_CHARS_FOR_NER,
}

with open(RAW_RUN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False)

print("wrote:", RAW_RUN_CONFIG_JSON)

## Utility Functions

In [5]:



def normalize_spaces(x: str) -> str:
    return re.sub(r"\s+", " ", str(x or "")).strip()


def strip_outer_punct(x: str) -> str:
    x = str(x or "").strip()
    x = x.strip(" \t\r\n-–—:;,.\'\"“”‘’()[]{}<>|/")
    return x


def split_camel(x: str) -> str:
    return normalize_spaces(re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(x or "")))


def pre_ner_text(x: str) -> str:
    return normalize_spaces(str(x or "")[:MAX_BLOCK_CHARS_FOR_NER])


def simple_title_case(x: str) -> str:
    words = []
    for w in str(x or "").split():
        if w.isupper() and len(w) <= 4:
            words.append(w)
        elif re.fullmatch(r"[A-Za-z]+", w):
            words.append(w.capitalize())
        else:
            words.append(w)
    return " ".join(words)


def normalize_entity_key(x: str) -> str:
    s = normalize_spaces(split_camel(str(x or "")).lower())
    s = re.sub(r"['’]", "", s)
    s = re.sub(r"[\(\)\[\]\{\}\.,;:!?\"“”‘’/\\|]+", " ", s)
    s = re.sub(r"[-_]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def canonicalize_surface(surface: str) -> str:
    s = normalize_spaces(split_camel(surface))
    s = strip_outer_punct(s)
    s = re.sub(r"^(the|a|an)\s+", "", s, flags=re.IGNORECASE)
    s = re.sub(r"(?:'s|’s)$", "", s, flags=re.IGNORECASE)
    s = normalize_spaces(s)
    key = normalize_entity_key(s)
    if key in SAFE_CANONICAL_MAP:
        return SAFE_CANONICAL_MAP[key]
    if re.fullmatch(r"[A-Z]{2,6}", s):
        return s
    if key in GENERIC_TECH_TERMS:
        if key == "large language models":
            return "Large Language Models"
        return simple_title_case(key)
    return simple_title_case(s)


def parquet_exists_and_nonempty(path: str) -> bool:
    return os.path.exists(path) and os.path.getsize(path) > 0


def validate_raw_schema(df: pd.DataFrame) -> None:
    required = [
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len", "entity_surface",
        "entity_surface_norm", "entity_key", "raw_label", "raw_score", "char_start", "char_end"
    ]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"Missing required raw mention columns: {missing}"


def empty_raw_mentions_frame() -> pd.DataFrame:
    cols = [
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len", "entity_surface",
        "entity_surface_norm", "entity_key", "raw_label", "raw_score", "char_start", "char_end"
    ]
    return pd.DataFrame(columns=cols)


def get_shard_path(shard_id: int) -> str:
    return os.path.join(RAW_MENTION_SHARDS_DIR, f"part_{shard_id:04d}_mentions.parquet")


def load_clean_ai_blocks() -> pd.DataFrame:
    blocks = pd.read_parquet(CLEAN_AI_BLOCKS_PARQUET)
    required_cols = {"doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain", "clean_block_text", "clean_block_len"}
    missing = required_cols - set(blocks.columns)
    assert not missing, f"Missing required block columns in clean_ai_blocks.parquet: {sorted(missing)}"
    if "p_ai_block" not in blocks.columns:
        blocks["p_ai_block"] = np.nan
    keep_cols = [
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len"
    ]
    blocks = blocks[keep_cols].copy()
    blocks["clean_block_text"] = blocks["clean_block_text"].fillna("").astype(str)
    blocks["block_uid"] = blocks["block_uid"].astype(str)
    blocks = blocks[blocks["clean_block_text"].str.len() > 0].drop_duplicates(["block_uid"]).reset_index(drop=True)
    return blocks


def load_or_init_manifest() -> pd.DataFrame:
    if os.path.exists(RAW_MANIFEST_CSV):
        manifest = pd.read_csv(RAW_MANIFEST_CSV)
    else:
        manifest = pd.DataFrame(columns=["shard_id", "out_path", "n_blocks", "n_mentions", "status"])
    expected_cols = ["shard_id", "out_path", "n_blocks", "n_mentions", "status"]
    for col in expected_cols:
        if col not in manifest.columns:
            manifest[col] = pd.Series(dtype="object")
    return manifest[expected_cols].copy()


def save_manifest(manifest: pd.DataFrame) -> None:
    manifest = manifest.sort_values("shard_id").reset_index(drop=True)
    manifest.to_csv(RAW_MANIFEST_CSV, index=False)


def extract_entities_from_frame(model: GLiNER, pdf: pd.DataFrame) -> pd.DataFrame:
    texts = [pre_ner_text(x) for x in pdf["clean_block_text"].tolist()]
    preds = model.batch_predict_entities(texts, NER_LABELS, threshold=GLINER_THRESHOLD)
    records = []
    for row, ents in zip(pdf.itertuples(index=False), preds):
        for ent in ents:
            surface = str(ent.get("text", "")).strip()
            if not surface:
                continue
            raw_label = str(ent.get("label", "")).strip().lower()
            score = float(ent.get("score", 0.0))
            start = ent.get("start", None)
            end = ent.get("end", None)
            records.append({
                "doc_id": int(row.doc_id),
                "block_id": int(row.block_id),
                "block_uid": str(row.block_uid),
                "url": row.url,
                "date": row.date,
                "language": row.language,
                "title": row.title,
                "domain": row.domain,
                "clean_block_text": row.clean_block_text,
                "p_ai_block": row.p_ai_block,
                "clean_block_len": int(row.clean_block_len),
                "entity_surface": surface,
                "entity_surface_norm": canonicalize_surface(surface),
                "entity_key": normalize_entity_key(surface),
                "raw_label": raw_label,
                "raw_score": score,
                "char_start": int(start) if start is not None else -1,
                "char_end": int(end) if end is not None else -1,
            })
    if not records:
        return empty_raw_mentions_frame()
    out = pd.DataFrame(records)
    return out[[
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len", "entity_surface",
        "entity_surface_norm", "entity_key", "raw_label", "raw_score", "char_start", "char_end"
    ]]

## Raw NER Reuse / Merge / GLiNER Extraction

In [6]:
blocks = load_clean_ai_blocks()
print("clean_ai_blocks shape:", blocks.shape)
display(blocks.head(3))

planned_shards = []
for shard_id, start_row in enumerate(range(0, len(blocks), NER_BLOCKS_PER_SHARD)):
    end_row = min(start_row + NER_BLOCKS_PER_SHARD, len(blocks))
    planned_shards.append((shard_id, start_row, end_row))

manifest = load_or_init_manifest()
existing_shard_files = sorted([f for f in os.listdir(RAW_MENTION_SHARDS_DIR) if f.endswith(".parquet")])
existing_shard_ids = {int(re.search(r"part_(\d+)_mentions\.parquet$", x).group(1)) for x in existing_shard_files if re.search(r"part_(\d+)_mentions\.parquet$", x)}
planned_shard_ids = {x[0] for x in planned_shards}
missing_shard_ids = sorted(planned_shard_ids - existing_shard_ids)

need_raw_ner = FORCE_RERUN_RAW_NER or ((not parquet_exists_and_nonempty(RAW_MENTIONS_PARQUET)) and len(missing_shard_ids) > 0)
need_merge_only = FORCE_REBUILD_RAW_FROM_SHARDS or ((not parquet_exists_and_nonempty(RAW_MENTIONS_PARQUET)) and len(missing_shard_ids) == 0)

print("planned shards:", len(planned_shards))
print("existing shard files:", len(existing_shard_ids))
print("missing shard ids:", len(missing_shard_ids))
print("need_raw_ner:", need_raw_ner)
print("need_merge_only:", need_merge_only)

if need_raw_ner:
    assert DEVICE == "cuda", "GPU is required for GLiNER raw extraction."
    ner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME).to(DEVICE)

    for shard_id, start_row, end_row in tqdm(planned_shards, desc="GLiNER shard extraction"):
        out_path = get_shard_path(shard_id)
        if (not FORCE_RERUN_RAW_NER) and os.path.exists(out_path):
            continue

        shard_df = blocks.iloc[start_row:end_row].copy().reset_index(drop=True)
        all_out = []
        n_batches = math.ceil(len(shard_df) / NER_BATCH_SIZE)

        for batch_idx in tqdm(range(n_batches), total=n_batches, desc=f"part_{shard_id:04d}", leave=False):
            sub = shard_df.iloc[batch_idx * NER_BATCH_SIZE:(batch_idx + 1) * NER_BATCH_SIZE].copy()
            part_out = extract_entities_from_frame(ner_model, sub)
            if len(part_out) > 0:
                all_out.append(part_out)

        shard_mentions = pd.concat(all_out, ignore_index=True) if all_out else empty_raw_mentions_frame()
        shard_mentions.to_parquet(out_path, index=False)

        manifest = manifest.loc[manifest["shard_id"] != shard_id].copy()
        manifest = pd.concat([
            manifest,
            pd.DataFrame([{
                "shard_id": shard_id,
                "out_path": out_path,
                "n_blocks": len(shard_df),
                "n_mentions": len(shard_mentions),
                "status": "ok",
            }])
        ], ignore_index=True)
        save_manifest(manifest)

        del shard_df, all_out, shard_mentions
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del ner_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if need_raw_ner or need_merge_only:
    shard_files = sorted([
        os.path.join(RAW_MENTION_SHARDS_DIR, f)
        for f in os.listdir(RAW_MENTION_SHARDS_DIR)
        if f.endswith(".parquet")
    ])
    assert shard_files, f"No raw mention shards found in: {RAW_MENTION_SHARDS_DIR}"

    frames = []
    for fpath in tqdm(shard_files, desc="Loading raw mention shards"):
        frames.append(pd.read_parquet(fpath))
    raw_mentions = pd.concat(frames, ignore_index=True) if frames else empty_raw_mentions_frame()
    del frames
    gc.collect()

    validate_raw_schema(raw_mentions)
    raw_mentions.to_parquet(RAW_MENTIONS_PARQUET, index=False)
    print("rebuilt raw mentions to:", RAW_MENTIONS_PARQUET)
else:
    assert parquet_exists_and_nonempty(RAW_MENTIONS_PARQUET), (
        "Missing raw mention parquet and shard rebuild path is unavailable."
    )
    raw_mentions = pd.read_parquet(RAW_MENTIONS_PARQUET)
    print("loaded raw mentions from:", RAW_MENTIONS_PARQUET)

validate_raw_schema(raw_mentions)
manifest = load_or_init_manifest()
print("raw_mentions shape:", raw_mentions.shape)
print("manifest shape:", manifest.shape)
display(raw_mentions.head(5))
display(manifest.head(10))

clean_ai_blocks shape: (731989, 11)


,doc_id,block_id,block_uid,url,date,language,title,domain,clean_block_text,p_ai_block,clean_block_len
0,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,1977
1,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542
2,1,8,1:8,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,Imagine showing these to a tribe that's never ...,0.998448,557


planned shards: 30
existing shard files: 30
missing shard ids: 0
need_raw_ner: False
need_merge_only: False
loaded raw mentions from: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04A_raw_mentions/entity_mentions_raw.parquet
raw_mentions shape: (5113052, 18)
manifest shape: (30, 5)


,doc_id,block_id,block_uid,url,date,language,title,domain,clean_block_text,p_ai_block,clean_block_len,entity_surface,entity_surface_norm,entity_key,raw_label,raw_score,char_start,char_end
0,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,1977,Byron Gilliam,Byron Gilliam,byron gilliam,person,0.903304,1035,1048
1,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,1977,Carlos Gonzalez Campo,Carlos Gonzalez Campo,carlos gonzalez campo,person,0.646446,1447,1468
2,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542,AI,AI,ai,technology,0.811913,70,72
3,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542,AI,AI,ai,technology,0.597019,101,103
4,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542,AI,AI,ai,technology,0.868772,175,177


,shard_id,out_path,n_blocks,n_mentions,status
0,0,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,174687,ok
1,1,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,174681,ok
2,2,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,171504,ok
3,3,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,175129,ok
4,4,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,173717,ok
5,5,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,172515,ok
6,6,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,174042,ok
7,7,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,175265,ok
8,8,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,175312,ok
9,9,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,172452,ok
